<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [ ]:
!pip install datasets

Импорт необходимых библиотек

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [ ]:
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')  # NLTK 3.8+


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [ ]:
# Загрузим датасет
dataset = load_dataset('imdb')

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [ ]:
sentences = []
word_threshold = 32

# Разбиваем каждый отзыв на предложения; одно предложение — один пример.
# Оставляем только короткие предложения (меньше word_threshold слов).
for text in tqdm(dataset['train']['text']):
    for sent in sent_tokenize(text):
        toks = word_tokenize(sent.lower())
        if 0 < len(toks) < word_threshold:
            sentences.append(sent)


In [ ]:
print("Всего предложений:", len(sentences))

Посчитаем для каждого слова его встречаемость.

In [ ]:
words = Counter()

for sent in tqdm(sentences):
    for w in word_tokenize(sent.lower()):
        words[w] += 1


Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [ ]:
vocab_size = 40000
special_tokens = ['<unk>', '<bos>', '<eos>', '<pad>']

# Топ vocab_size слов по частоте (без дубликатов со служебными)
top_words = []
seen = set(special_tokens)
for w, _ in words.most_common():
    if w in seen:
        continue
    top_words.append(w)
    seen.add(w)
    if len(top_words) >= vocab_size:
        break

vocab = set(special_tokens + top_words)


In [ ]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [ ]:
print("Всего слов в словаре:", len(vocab))

### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [ ]:
# Фиксированный порядок индексов: сначала служебные, затем топ по частоте
ordered_vocab = special_tokens + top_words
word2ind = {w: i for i, w in enumerate(ordered_vocab)}
ind2word = {i: w for w, i in word2ind.items()}


In [ ]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        sent = self.data[idx]
        toks = word_tokenize(sent.lower())
        ids = [self.bos_id] + [word2ind.get(t, self.unk_id) for t in toks] + [self.eos_id]
        return ids

    def __len__(self) -> int:
        return len(self.data)


In [ ]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> dict:
    # Копируем последовательности, чтобы не портить данные датасета при паддинге
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        seq = list(sequence)
        seq.extend([pad_id] * (max_seq_len - len(seq)))
        new_batch.append(seq)

    sequences = torch.LongTensor(new_batch).to(device)

    return {
        'input_ids': sequences[:, :-1],
        'target_ids': sequences[:, 1:],
    }


In [ ]:
# 80% train, от оставшихся 50/50 eval и test
train_sentences, temp = train_test_split(sentences, test_size=0.2, random_state=42)
eval_sentences, test_sentences = train_test_split(temp, test_size=0.5, random_state=42)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size, shuffle=True)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)


## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [ ]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexities = []
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids'])  # (batch, seq_len, vocab_size)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                batch['target_ids'].reshape(-1),
            )
            perplexities.append(torch.exp(loss).item())

    return sum(perplexities) / len(perplexities)


### Train loop (1 балл)

Напишите функцию для обучения модели.

In [ ]:
def train_model(model, criterion, optimizer, train_dataloader, num_epochs: int = 3):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        n_batches = 0
        for batch in tqdm(train_dataloader, desc=f'epoch {epoch+1}/{num_epochs}'):
            optimizer.zero_grad()
            logits = model(batch['input_ids'])
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                batch['target_ids'].reshape(-1),
            )
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
        avg_loss = running_loss / max(n_batches, 1)
        print(f'Epoch {epoch+1}: train loss = {avg_loss:.4f}')
    return model


### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [ ]:
class LanguageModel(nn.Module):
    """Языковая модель: эмбеддинги + LSTM + линейный слой по словарю."""

    def __init__(self, vocab_size: int, embed_dim: int = 256, hidden_dim: int = 512,
                 num_layers: int = 1, dropout: float = 0.3, pad_idx: int = 0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        emb = self.dropout(self.embedding(input_batch))
        out, _ = self.lstm(emb)
        return self.fc(out)


In [ ]:
pad_idx = word2ind['<pad>']
vocab_n = len(word2ind)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

model_v1 = LanguageModel(
    vocab_size=vocab_n, embed_dim=256, hidden_dim=512, num_layers=1, dropout=0.3, pad_idx=pad_idx
).to(device)
optimizer_v1 = torch.optim.Adam(model_v1.parameters(), lr=1e-3)

train_model(model_v1, criterion, optimizer_v1, train_dataloader, num_epochs=3)

test_ppl_v1 = evaluate(model_v1, criterion, test_dataloader)
eval_ppl_v1 = evaluate(model_v1, criterion, eval_dataloader)
print(f'Эксперимент 1 (LSTM x1): eval perplexity ~ {eval_ppl_v1:.2f}, test ~ {test_ppl_v1:.2f}')


### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [ ]:
# Эксперимент 2: GRU, 2 слоя, другой dropout и AdamW

class LanguageModelGRU(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 256, hidden_dim: int = 384,
                 num_layers: int = 2, dropout: float = 0.4, pad_idx: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        emb = self.dropout(self.embedding(input_batch))
        out, _ = self.gru(emb)
        return self.fc(out)

model_v2 = LanguageModelGRU(
    vocab_size=vocab_n, embed_dim=256, hidden_dim=384, num_layers=2, dropout=0.4, pad_idx=pad_idx
).to(device)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=3e-4, weight_decay=1e-4)

train_model(model_v2, criterion, optimizer_v2, train_dataloader, num_epochs=3)

test_ppl_v2 = evaluate(model_v2, criterion, test_dataloader)
eval_ppl_v2 = evaluate(model_v2, criterion, eval_dataloader)
print(f'Эксперимент 2 (GRU x2 + AdamW): eval perplexity ~ {eval_ppl_v2:.2f}, test ~ {test_ppl_v2:.2f}')


### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

### Выводы (отчёт)

| Эксперимент | Архитектура | Оптимизатор | Комментарий |
|-------------|-------------|-------------|-------------|
| 1 | LSTM, 1 слой, hidden=512, dropout=0.3 | Adam lr=1e-3 | базовая модель |
| 2 | GRU, 2 слоя, hidden=384, dropout=0.4 | AdamW | другой тип RNN, сильнее dropout, weight decay |

**Перплексия** (чем ниже, тем лучше) смотрите в выводе ячеек обучения (eval и test). Для языковой модели сравниваем числа после обучения на одних и тех же данных.

**Идеи улучшения:** больше эпох и ранняя остановка по val perplexity; подбор learning rate; `clip_grad_norm_`; больший `embed_dim` / `hidden_dim`; scheduler (ReduceLROnPlateau); для очень больших словарей — разбиение выходного слоя (см. adaptive softmax).
